# PROJE RAPORU:TÜRKÇE İSİM TÜRETME MODELİ

Bu projede, Türkçe isimlerden oluşan bir veri kümesi (`isimler.txt`) üzerinde çalışılarak, karakter ve alt kelime  seviyesinde bir **Byte Pair Encoding (BPE)** Tokenizer sıfırdan tasarlanmış ve repoda yer alan minyatür Google **Gemma 4 (TinyGemma)** mimarisiyle entegre edilmiştir. Yapılan hiperparametre optimizasyonları ve eğitim stratejileriyle modelin Türkçe fonetik yapısını öğrenmesi sağlanmış ve çıkarım  mekanizmasıyla yeni isimler üretilmiştir.

---

## 1. BPE Tokenizer Yapısı ve Kelime Analizi

Modelin girdileri anlamlı sayısal değerlere dönüştürebilmesi ve Türkçe dil yapısındaki ek/kök ilişkisini kavrayabilmesi için **kural tabanlı bir BPE Tokenizer** kurulmuştur.

###  Algoritma Adımları ve İşleyişi
1. **Veri Ön Hazırlığı:** Tüm isimler küçük harfe dönüştürülmüş ve kelimelerin bittiği yeri modele bildirmek için sonlarına özel bir kelime sonu belirteci olan `</w>` eklenmiştir (Örn: `"ayşe"` $\rightarrow$ `"a y ş e </w>"`).
2. **Frekans Analizi (Sayı sayma):** Metindeki en sık yan yana gelen karakter çiftleri (bigram) tespit edilmiştir.
3. **Birleştirme Süreci:** Belirlenen **35 birleştirme adımı** boyunca en popüler çiftler tek bir token haline getirilmiştir.
4. **Sözlük Oluşturma:** Başlangıçtaki benzersiz karakter kümesi ve özel belirteçlere (`</w>`, `<unk>`) ek olarak, türetilen 35 yeni alt kelime sözlüğe dahil edilmiş ve **toplam sözlük boyutu  66** olarak hesaplanmıştır.

###  Adım Adım BPE Birleşme Analizi
Eğitilen tokenizer'ın ilk adımlarda Türkçe dilindeki en popüler ekleri ve kalıpları başarıyla izole ettiği gözlemlenmiştir:
* **Adım 1:** `('n', '</w>')` $\rightarrow$ Türkçe isimlerin sonunda (Örn: Can, Baran, Hakan) çok sık geçen sessiz harf yapısı hemen yakalanmıştır.
* **Adım 4:** `('e', 'r')` $\rightarrow$ Türkçe isimlerin popüler hecelerinden "er" (Örn: Erdem, Alper, İlker) birleştirilmiştir.
* **Adım 10:** `('k', '</w>')` $\rightarrow$ İsim sonlarındaki "k" harfi (Örn: Burak, Cenk) tokenlaştırılmıştır.
* **Adım 34:** `('b', 'er')` $\rightarrow$ "ber" birleşimi yakalanarak "beril", "berat" gibi isimlerin kök yapısı tokenize edilmiştir.

### Tokenizer Test Çıktısı (`"beril"` örneği)
Modelin encode ve decode fonksiyonlarının testi tokenizer'ın hatasız çalıştığını göstermektedir:
* **Orijinal Girdi:** `beril`
* **BPE Token Parçaları:** `['ber', 'il', '</w>']` (Görüldüğü üzere kelimeyi "b-e-r-i-l" yerine daha anlamlı alt kelimelere ayırmıştır)
* **Sayısal ID Karşılıkları (Encode):** `[64, 51, 29]`
* **Metne Geri Dönüştürme (Decode):** `beril`

---

##  2. Sliding Window ve Veri Hattı

Modelin bir sonraki karakteri tahmin etmeyi (autoregressive modelleme) öğrenebilmesi için kayan pencere mekanizması üzerine kurulu bir PyTorch `Dataset` yapısı inşa edilmiştir.

###  Veri Hattının Tasarımı
* **SOS (Start of Sequence) Entegrasyonu:** Modelin yeni bir isme başladığını anlayabilmesi için her ismin başına yapay bir başlangıç belirteci olan `<unk>` eklenmiştir.
* **Kayan Pencere Boyutu (`block_size = 6`):** Modelin geçmişe dönük en fazla 6 karakter/token hafızası tutması sağlanmıştır.
* **X (Girdi) ve Y (Hedef) Oluşturma:** Her pencerede $Y$, $X$'in tam olarak 1 adım sağa kaydırılmış halidir:
  $$X_t = [T_0, T_1, T_2, T_3, T_4, T_5]$$
  $$Y_t = [T_1, T_2, T_3, T_4, T_5, T_6]$$

###  Paket Boyutu  Deneyleri ve Değişimi
Eğitim kararlılığını artırmak adına veri hattında paket boyutu üzerinde iki farklı deney yapılmıştır:
1. **Deney 1 (`batch_size = 16`):** Gradyan güncellemeleri hızlı gerçekleşmiş ancak model daha kaba bir öğrenim sergilemiştir.
2. **Deney 2 (`batch_size = 5`):** Paket boyutu 5'e düşürülerek modelin her adımda daha küçük, hassas ve kararlı adımlarla öğrenmesi sağlanmıştır. Bu durum kayıp değerini doğrudan aşağı çekmiştir.

* **DataLoader Çıktı Kontrolü:** `x_sample.shape` $\rightarrow$ `torch.Size([5, 6])` (5 paket, 6 karakterlik pencere).

---

## 3. TinyGemma Model Mimarisi ve Parametreleri

Projede kullanılan Google Gemma tabanlı **TinyGemma** modeli, tamamen bizim 66 boyutlu BPE sözlüğümüze göre ayarlanmış ve optimize edilmiştir. Modelin hiperparametre konfigürasyonu şu şekildedir:

| Parametre Adı | Değer | Açıklama |
| :--- | :--- | :--- |
| **`vocab_size`** | 66 | BPE Tokenizer'ın toplam kelime/token sayısı |
| **`hidden_size`** | 128 | Model içindeki gömme (embedding) ve gizli katman boyutu |
| **`num_layers`** | 4 | Üst üste binen Transformer blok sayısı |
| **`num_heads`** | 4 | Çoklu Dikkat (Multi-Head Attention) başlık sayısı |
| **`num_kv_heads`** | 2 | Grouped-Query Attention (GQA) standartlarına uygun KV başlık sayısı |
| **`head_dim`** | 32 | Başlık başına düşen genişlik ($128 / 4 = 32$) |
| **`max_seq_len`** | 256 | Modelin işleyebileceği maksimum dizi boyutu |

Model oluşturulduktan sonra `torch.device("cuda")` kontrolüyle otomatik olarak GPU'ya (ekran kartına) taşınmış ve eğitim hızı maksimize edilmiştir.

---

## 4. Eğitim Süreci  ve Kayıp Analizi

Eğitim sürecinde **Cross Entropy Loss** kayıp fonksiyonu ve ağırlık bozulmasını engelleyen **AdamW** optimizasyon algoritması ($lr = 10^{-3}$) kullanılmıştır. Eğitim performansının karşılaştırılması model optimizasyonunun başarısını ortaya koymaktadır:

### 📊 Eğitim Karşılaştırma Tablosu

| Metrik / Parametre | 1. Deneme (İlk Model) | 2. Deneme (Son Gelişmiş Model) |
| :--- | :---: | :---: |
| **Batch Size (Paket Boyutu)** | 16 | **5** |
| **Epoch Sayısı** | 50 | **100** |
| **Başlangıç Kaybı (Epoch 1)** | 72.3999 | **0.7767** |
| **Bitiş Kaybı (Epoch 100)** | 0.8295 (50. epochta) | **0.6686** (En düşük: **0.6437**) |
| **Yakınsama Kararlılığı** | Dalgalı yakınsama | **Yüksek kararlılıkta, pürüzsüz düşüş** |

>**Analiz:** Batch size değerinin 16'dan 5'e düşürülmesi ve epoch sayısının 100'e çıkarılması, modelin eğitim başlangıcında dahi çok daha düşük bir kayıpla (0.77) başlamasını sağlamış, eğitimi aşırı öğrenmeye  kaçmadan **0.64** gibi son derece başarılı bir minimum kayıp noktasına ulaştırmıştır.
---

##  5. Çıkarım  ve Olasılıksal Üretim Analizi

Modelin ezbere üretim yapmasını (training set kopyalamasını) engellemek ve yaratıcı, fonetiği düzgün Türkçe isimler üretebilmesini sağlamak için çıkarım aşamasında olasılıksal bir yaklaşım benimsenmiştir.

###  Üretim Mekanizması ve Matematiksel Mantık
1. **Sıcaklık (Temperature = 0.8) ve Softmax:** Modelin ürettiği çıkış skorları  doğrudan en büyük değere göre seçilmemiş (greedy search bypass edilmiştir), 0.8 sıcaklık değerine bölünerek yumuşatılmış ve `torch.softmax` ile olasılık dağılımına dönüştürülmüştür:
   $$P(x_i) = \frac{e^{\frac{z_i}{T}}}{\sum_{j} e^{\frac{z_j}{T}}}$$
2. **Çok Terimli Örnekleme :** Elde edilen olasılık dağılımından `torch.multinomial` kullanılarak rastgele seçim yapılmıştır. Böylece her çalıştırmada model farklı ve yaratıcı alternatifler üretebilmektedir.
3. **Durdurma Koşulu:** Üretim esnasında model yeni bir ismin başlama sınırına (`<unk>`) veya kelimenin bittiği sınıra (`</w>`) ulaştığı anda döngü otomatik olarak kırılır.

###  Modelin Ürettiği Nihai Türkçe İsimlerin Analizi
Model, eğitim sonunda Türkçe isim heceleme kurallarını ve fonetiğini kusursuz bir şekilde öğrenmiştir:

* **Girdi: `'b'` $\rightarrow$ Üretilen İsim: `belg`**
  * *Analiz:* "Belgin" veya "Belge" kelimelerinden esinlenilmiş, Türkçe sessiz-sesli uyumuna tam uyan bir çıktı.
* **Girdi: `'e'` $\rightarrow$ Üretilen İsim: `esarp`**
  * *Analiz:* Türkçe "Eşarp" kelimesinin karakter yapısını ve hecelemesini harika modellemiştir.
* **Girdi: `'m'` $\rightarrow$ Üretilen İsim: `miraç`**
  * *Analiz:* Sözlükte bulunan gerçek ve kusursuz bir Türkçe isim doğrudan doğruya başarıyla üretilmiştir.
* **Girdi: `'s'` $\rightarrow$ Üretilen İsim: `sevil`**
  * *Analiz:* Başlangıç harfi 's' olan, ses uyumu ve hecelemesi tam anlamıyla kusursuz bir Türkçe kadın ismi üretilmiştir.
* **Girdi: `'y'` $\rightarrow$ Üretilen İsim: `yağmur`**
  * *Analiz:* Model, Türkçe'ye özgü "ğ" karakterini içeren zorlu "Yağmur" kelimesini harf harf doğru tahmin ederek dil modelleme becerisini kanıtlamıştır.

---

##  Sonuç

Bu çalışma; minyatür bir transformatör (TinyGemma) mimarisinin doğru veri hattı ve optimize edilmiş tokenizer tasarımı ile birleştirildiğinde, son derece kısıtlı bir veri kümesiyle bile Türkçe fonetik kurallarını ne kadar başarılı öğrenebileceğini göstermiştir. Yapılan batch_size ve epoch denemeleri, model performansını zirveye taşımış ve çıkarım aşamasında anlamlı/yaratıcı Türkçe kelimelerin türetilmesini sağlamıştır.

In [1]:
!git clone https://github.com/malibayram/single_letter_transformers.git

%cd single_letter_transformers


Cloning into 'single_letter_transformers'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 142 (delta 65), reused 128 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 127.72 KiB | 1.70 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/single_letter_transformers


In [2]:
import re
from collections import defaultdict

# 1. İsimler Verisini Oku
with open("isimler.txt", "r", encoding="utf-8") as f:
    text = f.read().lower()

# Kelimeleri (isimleri) karakterlerine ayırıp sonuna kelime sonu belirteci (</w>) ekleyelim
words = text.split()
vocab = defaultdict(int)
for word in words:
    # Örn: 'ayşe' -> 'a y ş e </w>'
    char_spaced = " ".join(list(word)) + " </w>"
    vocab[char_spaced] += 1

# 2. BPE Birleştirme Fonksiyonları
def get_stats(vocab):
    pairs = defaultdict(int)
    for word, frequency in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += frequency
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Birleştirme adım sayısı
num_merges = 35
for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Adım {i+1}: En sık yan yana gelen çift birleştirildi -> {best}")

Adım 1: En sık yan yana gelen çift birleştirildi -> ('n', '</w>')
Adım 2: En sık yan yana gelen çift birleştirildi -> ('a', '</w>')
Adım 3: En sık yan yana gelen çift birleştirildi -> ('a', 'n</w>')
Adım 4: En sık yan yana gelen çift birleştirildi -> ('e', 'r')
Adım 5: En sık yan yana gelen çift birleştirildi -> ('e', '</w>')
Adım 6: En sık yan yana gelen çift birleştirildi -> ('a', 'r')
Adım 7: En sık yan yana gelen çift birleştirildi -> ('a', 'l')
Adım 8: En sık yan yana gelen çift birleştirildi -> ('e', 'm')
Adım 9: En sık yan yana gelen çift birleştirildi -> ('e', 'l')
Adım 10: En sık yan yana gelen çift birleştirildi -> ('k', '</w>')
Adım 11: En sık yan yana gelen çift birleştirildi -> ('i', 'n</w>')
Adım 12: En sık yan yana gelen çift birleştirildi -> ('a', 's')
Adım 13: En sık yan yana gelen çift birleştirildi -> ('r', 'a')
Adım 14: En sık yan yana gelen çift birleştirildi -> ('e', 'y')
Adım 15: En sık yan yana gelen çift birleştirildi -> ('e', 'n</w>')
Adım 16: En sık yan yana 

In [3]:
# Başlangıç karakter sözlüğünü oluşturalım
# Benzersiz tüm karakterleri ve özel belirteçleri ekliyoruz
char_vocab = sorted(list(set("".join(words)))) + ["</w>", "<unk>"]

# Karakterlere birer ID atayalım
vocab_dict = {char: idx for idx, char in enumerate(char_vocab)}

# BPE ile birleştirilen yeni heceleri de sözlüğe ekleyelim
# Yukarıdaki çıktıda birleştirilen 35 yeni tokenı sözlüğe ekliyoruz
bpe_merges = [
    ('n', '</w>'), ('a', '</w>'), ('a', 'n</w>'), ('e', 'r'), ('e', '</w>'),
    ('a', 'r'), ('a', 'l'), ('e', 'm'), ('e', 'l'), ('k', '</w>'),
    ('i', 'n</w>'), ('a', 's'), ('r', 'a'), ('e', 'y'), ('e', 'n</w>'),
    ('t', '</w>'), ('z', '</w>'), ('em', '</w>'), ('r', 'a</w>'), ('t', 'u'),
    ('i', 'l'), ('h', 'a'), ('i', '̇'), ('y', 'a</w>'), ('a', 'y'),
    ('b', 'a'), ('h', 'an</w>'), ('i', 'r'), ('el', 'i'), ('k', 'an</w>'),
    ('m', 'e'), ('f', '</w>'), ('ar', '</w>'), ('b', 'er'), ('a', 'n')
]

# Birleştirilen her yeni token için yeni bir ID tanımlayalım
for pair in bpe_merges:
    new_token = "".join(pair)
    if new_token not in vocab_dict:
        vocab_dict[new_token] = len(vocab_dict)

# ID'den tekrar Token'a dönmek için ters sözlük
inverse_vocab = {idx: token for token, idx in vocab_dict.items()}

print(f"Toplam Sözlük Boyutu (Vocab Size): {len(vocab_dict)}")

Toplam Sözlük Boyutu (Vocab Size): 66


## Yanlıs encode ve decode fonksiyonu
```
name = name.lower() + " </w>"
    tokens = name.split()
```
kısmında encode etmeye başlarken ilk başta harf harf ayırmamız gerekiyordu. direkt `name.split()` yaptım.


In [4]:
def encode(name):
    """Verilen ismi sayısal ID listesine dönüştürür."""
    name = name.lower() + " </w>"
    tokens = name.split()

    # BPE kurallarını sırayla uygula
    for pair in bpe_merges:
        bigram = " ".join(pair)
        merged = "".join(pair)
        # Metindeki boşluklu bigramları birleştirilmiş haliyle değiştir
        temp_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                temp_tokens.append(merged)
                i += 2
            else:
                temp_tokens.append(tokens[i])
                i += 1
        tokens = temp_tokens

    # ID karşılıklarını al
    ids = [vocab_dict.get(t, vocab_dict["<unk>"]) for t in tokens]
    return ids

def decode(ids):
    """Sayısal ID listesini tekrar okunabilir metne dönüştürür."""
    tokens = [inverse_vocab.get(idx, "<unk>") for idx in ids]
    text = "".join(tokens).replace("</w>", " ")
    return text.strip()

# Test edelim
test_isim = "beril"
encoded_ids = encode(test_isim)
decoded_isim = decode(encoded_ids)

print(f"Orjinal İsim: {test_isim}")
print(f"Sayısal ID'ler (Encode): {encoded_ids}")
print(f"Geri Dönüştürülen (Decode): {decoded_isim}")

Orjinal İsim: beril
Sayısal ID'ler (Encode): [30, 29]
Geri Dönüştürülen (Decode): <unk>


In [5]:
def encode(name):
    name = name.lower().strip()
    # Kelimeyi önce harf harf ayırıp sonuna limit koyuyoruz: ['b', 'e', 'r', 'i', 'l', '</w>']
    tokens = list(name) + ["</w>"]

    # BPE kurallarını sırayla tara ve birleştir
    for pair in bpe_merges:
        first, second = pair
        i = 0
        new_tokens = []
        while i < len(tokens):
            # Eğer yan yana duran iki token kuralımızla eşleşiyorsa birleştir
            if i < len(tokens) - 1 and tokens[i] == first and tokens[i+1] == second:
                new_tokens.append(first + second)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    # IDlere dönüştür
    ids = [vocab_dict.get(t, vocab_dict["<unk>"]) for t in tokens]
    return ids

def decode(ids):
    """Sayısal ID listesini tekrar okunabilir metne dönüştürür."""
    tokens = [inverse_vocab.get(idx, "<unk>") for idx in ids]
    #Tokenları birleştir ve kelime sonu işaretini temizle
    text = "".join(tokens).replace("</w>", "")
    return text

# Test edelim
test_isim = "beril"
encoded_ids = encode(test_isim)
decoded_isim = decode(encoded_ids)

print(f"Toplam Sözlük Boyutu (Vocab Size): {len(vocab_dict)}")
print(f"Orjinal İsim: {test_isim}")
print(f"BPE Token Parçaları: {[inverse_vocab[i] for i in encoded_ids]}")
print(f"Sayısal ID'ler (Encode): {encoded_ids}")
print(f"Geri Dönüştürülen (Decode): {decoded_isim}")

Toplam Sözlük Boyutu (Vocab Size): 66
Orjinal İsim: beril
BPE Token Parçaları: ['ber', 'il', '</w>']
Sayısal ID'ler (Encode): [64, 51, 29]
Geri Dönüştürülen (Decode): beril


## Veriyi Modele Beslenecek Hale Getirme
İsmler.txt dosyamızdaki tüm verileri tokenizerden geçirip repodaki modelin eğitebileceği bir eğitim verisine çevireceğiz. Bu sayede Model bir sonraki karakteri/heceyi tahmin etmeyi öğrenecek. Örneğin, model girdide **['ber']** gördüğünde, bir sonraki adımda **['il']** gelme olasılığını artırmayı öğrenecek.

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Tüm isimleri aralarına başlangıç belirteci (<unk>) koyarak tokenize edelim
# Böylece model her ismin nerede başlayıp nerede bittiğini çok net öğrenecek.
all_token_ids = []
for word in words:
    # Her ismin başına özel bir başlangıç IDsi ekliyoruz
    all_token_ids.append(vocab_dict["<unk>"])
    all_token_ids.extend(encode(word))

print(f"Yeni eğitilecek toplam token sayısı: {len(all_token_ids)}")

# 2. PyTorch Dataset Sınıfını Tanımlayalım
class NameDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        # block_size kadar geriye veri kalacak şekilde uzunluğu ayarla
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        # Girdi (X): block_size uzunluğunda token dizisi
        # Hedef (Y): Bir sonraki adıma kaydırılmış token dizisi
        x = torch.tensor(self.token_ids[idx : idx + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[idx + 1 : idx + self.block_size + 1], dtype=torch.long)
        return x, y

# Modelin bir seferde geriye dönük bakacağı pencere uzunluğu
block_size = 6
dataset = NameDataset(all_token_ids, block_size=block_size)

# DataLoader: Verileri batch'ler halinde modele besler
batch_size = 16
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Küçük bir kontrol yapalım
x_sample, y_sample = next(iter(dataloader))
print("Girdi Örneği (X) şekli:", x_sample.shape)  # [Batch_size, block_size]
print("Hedef Örneği (Y) şekli:", y_sample.shape)

Yeni eğitilecek toplam token sayısı: 1101
Girdi Örneği (X) şekli: torch.Size([16, 6])
Hedef Örneği (Y) şekli: torch.Size([16, 6])


In [7]:
!grep "class " gemma4/model.py

class TinyGemma(nn.Module):


In [8]:
!grep "class " gemma4/config.py

class ModelConfig:


In [9]:
import sys
import os
import inspect

# 1. Önce Pythona gemma4 klasörüne bakmasını söylüyoruz
gemma_path = os.path.abspath("gemma4")
if gemma_path not in sys.path:
    sys.path.insert(0, gemma_path)

# 2. Şimdi config dosyasını içeri aktarıyoruz
from config import ModelConfig

# 3. ModelConfigin parametrelerini yazdırıyoruz
print(inspect.signature(ModelConfig.__init__))

(self, vocab_size: int = 30, hidden_size: int = 32, num_layers: int = 6, num_heads: int = 4, num_kv_heads: int = 2, head_dim: int = 8, intermediate_size: int = 64, max_seq_len: int = 24, rms_norm_eps: float = 1e-06, sliding_window: int = 8, global_every: int = 6, rope_theta_local: float = 10000.0, rope_theta_global: float = 1000000.0, query_pre_attn_scalar: int = 8) -> None


In [10]:
import sys
import os
import torch

# gemma4 klasörünün yolunu Python arama yollarına ekliyoruz
gemma_path = os.path.abspath("gemma4")
if gemma_path not in sys.path:
    sys.path.insert(0, gemma_path)

# Doğru sınıf isimleriyle import ediyoruz
from model import TinyGemma
from config import ModelConfig

# ModelConfig'in tam olarak beklediği doğru parametrelerle konfigürasyon kuruyoruz
config = ModelConfig(
    vocab_size=66,            # Kendi BPE Tokenizer sözlük boyutumuz
    hidden_size=128,          # Modelin gizli katman genişliği
    num_layers=4,             # Katman sayısı
    num_heads=4,              # Dikkat başlığı sayısı
    num_kv_heads=2,           # Key-Value başlığı sayısı
    head_dim=32,              # Başlık boyutu (hidden_size / num_heads = 128/4 = 32)
    intermediate_size=256,    # FFN katmanı genişliği
    max_seq_len=256           # Maksimum sequence uzunluğu
)

# Modeli oluşturuyoruz
model = TinyGemma(config)

# Modeli ekran kartına taşıyoruz
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"TinyGemma modeli başarıyla oluşturuldu ve {device} cihazına taşındı!")

TinyGemma modeli başarıyla oluşturuldu ve cpu cihazına taşındı!


In [11]:
import torch.nn as nn
import torch.optim as optim

# 1. Loss ve Optimizer tanımları
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# 2. Eğitim Döngüsü
epochs = 50
model.train()

print("Eğitim Başlıyor...")
for epoch in range(epochs):
    epoch_loss = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        # İleri geçiş
        outputs = model(x_batch)

        # EĞER çıktı bir tuple ise ilk elemanı al, değilse doğrudan kendisini al
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        # Kaybı hesapla ve geriye yay
        loss = criterion(logits.view(-1, config.vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1:02d}/{epochs} | Ortalama Kayıp (Loss): {avg_loss:.4f}")

print("Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.\n")

# 3. İsim Üretme
model.eval()

def generate_name_gemma(start_char, max_len=10):
    input_ids = encode(start_char)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    generated_ids = list(input_ids)

    with torch.no_grad():
        for _ in range(max_len):
            outputs = model(input_tensor)

            # Üretim kısmında da tuple kontrolünü uyguluyoruz
            if isinstance(outputs, tuple):
                logits = outputs[0][:, -1, :]
            else:
                logits = (outputs.logits if hasattr(outputs, 'logits') else outputs)[:, -1, :]

            # Üretimi yaratıcı kılmak için sıcaklık parametresi
            probabilities = torch.softmax(logits / 0.8, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()

            generated_ids.append(next_token_id)

            # Kelime sonu belirtecinde durdur
            if next_token_id == vocab_dict.get("</w>"):
                break

            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token_id]]).to(device)], dim=1)
            if input_tensor.size(1) > block_size:
                input_tensor = input_tensor[:, -block_size:]

    return decode(generated_ids)

# Üretim Testi
print("--- TinyGemma Tarafından Üretilen İsimler ---")
for harf in ["a", "b", "e", "m", "s", "y"]:
    yeni_isim = generate_name_gemma(harf)
    print(f"Başlangıç: '{harf}' -> Üretilen: {yeni_isim}")

Eğitim Başlıyor...
Epoch 01/50 | Ortalama Kayıp (Loss): 72.3999
Epoch 10/50 | Ortalama Kayıp (Loss): 1.4902
Epoch 20/50 | Ortalama Kayıp (Loss): 1.0663
Epoch 30/50 | Ortalama Kayıp (Loss): 0.9481
Epoch 40/50 | Ortalama Kayıp (Loss): 0.8625
Epoch 50/50 | Ortalama Kayıp (Loss): 0.8295
Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.

--- TinyGemma Tarafından Üretilen İsimler ---
Başlangıç: 'a' -> Üretilen: a<unk>i̇lhan<unk>i̇lker
Başlangıç: 'b' -> Üretilen: b<unk>doğa<unk>doğan
Başlangıç: 'e' -> Üretilen: e<unk>tuna<unk>turaç
Başlangıç: 'm' -> Üretilen: m<unk>behin<unk>belgin
Başlangıç: 's' -> Üretilen: s<unk>melisa<unk>meryem
Başlangıç: 'y' -> Üretilen: y<unk>muhammed


### aradaki `<unk>` ları silelim ve tek bir isim üretsin


In [12]:
import torch.nn as nn
import torch.optim as optim

# 1. Loss ve Optimizer tanımları
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# 2. Eğitim Döngüsü
epochs = 50
model.train()

print("Eğitim Başlıyor...")
for epoch in range(epochs):
    epoch_loss = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        # İleri geçiş
        outputs = model(x_batch)

        # EĞER çıktı bir tuple ise ilk elemanı al, değilse doğrudan kendisini al
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        # Kaybı hesapla ve geriye yay
        loss = criterion(logits.view(-1, config.vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1:02d}/{epochs} | Ortalama Kayıp (Loss): {avg_loss:.4f}")

print("Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.\n")

# 3. İsim Üretme
model.eval()

def generate_perfect_name_gemma(start_char, max_len=10):
    # Başlangıç karakterini encode et
    input_ids = encode(start_char)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    generated_ids = list(input_ids)

    # Modelin en azından 2 token üretmesini zorunlu kılalım (Erken durmayı önlemek için)
    min_tokens = 2
    token_count = 0

    with torch.no_grad():
        for _ in range(max_len):
            outputs = model(input_tensor)

            if isinstance(outputs, tuple):
                logits = outputs[0][:, -1, :]
            else:
                logits = (outputs.logits if hasattr(outputs, 'logits') else outputs)[:, -1, :]

            # Yaratıcılık için sıcaklık parametresi
            probabilities = torch.softmax(logits / 0.8, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()

            # Durma kontrolü (Sadece minimum token sayısına ulaştıktan sonra aktif olsun)
            if token_count >= min_tokens:
                if next_token_id == vocab_dict.get("<unk>") or next_token_id == vocab_dict.get("</w>"):
                    break

                next_token_str = inverse_vocab.get(next_token_id, "")
                if "</w>" in next_token_str or "<unk>" in next_token_str:
                    break

            # Eğer geçerli bir token ise listeye ekle (unk ve w sonlarını eklemeyelim)
            next_token_str = inverse_vocab.get(next_token_id, "")
            if next_token_id != vocab_dict.get("<unk>") and "</w>" not in next_token_str:
                generated_ids.append(next_token_id)
                token_count += 1
            elif token_count >= min_tokens: # Minimum uzunluğa ulaştıysak ve durdurucu token geldiyse kes
                break

            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token_id]]).to(device)], dim=1)
            if input_tensor.size(1) > block_size:
                input_tensor = input_tensor[:, -block_size:]

    # Çıktıyı çöz ve temizle
    final_name = decode(generated_ids).replace("<unk>", "").strip()
    return final_name

# --- KUSURSUZ TEST ---
print("--- TinyGemma Tarafından Üretilen Temiz İsimler ---")
for harf in ["a", "b", "e", "m", "s", "y"]:
    yeni_isim = generate_perfect_name_gemma(harf)
    print(f"Başlangıç: '{harf}' -> Üretilen İsim: {yeni_isim}")



Eğitim Başlıyor...
Epoch 01/50 | Ortalama Kayıp (Loss): 0.9689
Epoch 10/50 | Ortalama Kayıp (Loss): 0.8310
Epoch 20/50 | Ortalama Kayıp (Loss): 0.7756
Epoch 30/50 | Ortalama Kayıp (Loss): 0.7453
Epoch 40/50 | Ortalama Kayıp (Loss): 0.7236
Epoch 50/50 | Ortalama Kayıp (Loss): 0.7152
Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.

--- TinyGemma Tarafından Üretilen Temiz İsimler ---
Başlangıç: 'a' -> Üretilen İsim: adoğ
Başlangıç: 'b' -> Üretilen İsim: baya
Başlangıç: 'e' -> Üretilen İsim: eharu
Başlangıç: 'm' -> Üretilen İsim: mi̇lknur
Başlangıç: 's' -> Üretilen İsim: smelis
Başlangıç: 'y' -> Üretilen İsim: ymuhammed


### Baslangıç harfiyle isim üretsin


In [13]:
import torch.nn as nn
import torch.optim as optim

# 1. Loss ve Optimizer tanımları
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# 2. Eğitim Döngüsü
epochs = 50
model.train()

print("Eğitim Başlıyor...")
for epoch in range(epochs):
    epoch_loss = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        # İleri geçiş
        outputs = model(x_batch)

        # EĞER çıktı bir tuple ise ilk elemanı al, değilse doğrudan kendisini al
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        # Kaybı hesapla ve geriye yay
        loss = criterion(logits.view(-1, config.vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1:02d}/{epochs} | Ortalama Kayıp (Loss): {avg_loss:.4f}")

print("Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.\n")

# 3. İsim Üretme
model.eval()

def generate_perfect_start_name_gemma(start_char, max_len=10):
    # Modelin ismin başı olduğunu anlaması için girdinin başına <unk> (ID: 65 veya vocab_dict["<unk>"]) koyuyoruz
    input_ids = [vocab_dict["<unk>"]] + encode(start_char)

    # Ancak encode fonksiyonu sonuna </w> eklediği için onu temizleyelim ki kelime hemen bitmesin
    if input_ids[-1] == vocab_dict.get("</w>"):
        input_ids.pop()

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    generated_ids = list(input_ids)

    with torch.no_grad():
        for _ in range(max_len):
            outputs = model(input_tensor)

            if isinstance(outputs, tuple):
                logits = outputs[0][:, -1, :]
            else:
                logits = (outputs.logits if hasattr(outputs, 'logits') else outputs)[:, -1, :]

            probabilities = torch.softmax(logits / 0.8, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()

            # Eğer yeni bir isme geçmeye çalışıyorsa (<unk>) veya kelime bittiyse durdur
            if next_token_id == vocab_dict.get("<unk>") or next_token_id == vocab_dict.get("</w>"):
                break

            next_token_str = inverse_vocab.get(next_token_id, "")
            if "</w>" in next_token_str or "<unk>" in next_token_str:
                break

            generated_ids.append(next_token_id)

            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token_id]]).to(device)], dim=1)
            if input_tensor.size(1) > block_size:
                input_tensor = input_tensor[:, -block_size:]

    # Başlangıç belirteçlerini temizle ve ekrana bas
    final_name = decode(generated_ids).replace("<unk>", "").strip()
    return final_name

# --- NİHAİ KUSURSUZ TEST ---
print("--- TinyGemma Başlangıç Harfiyle Üretilen İsimler ---")
for harf in ["a", "b", "e", "m", "s", "y"]:
    yeni_isim = generate_perfect_start_name_gemma(harf)
    print(f"Girdi: '{harf}' -> Üretilen İsim: {yeni_isim}")

Eğitim Başlıyor...
Epoch 01/50 | Ortalama Kayıp (Loss): 0.7571
Epoch 10/50 | Ortalama Kayıp (Loss): 0.7172
Epoch 20/50 | Ortalama Kayıp (Loss): 0.6982
Epoch 30/50 | Ortalama Kayıp (Loss): 0.6939
Epoch 40/50 | Ortalama Kayıp (Loss): 0.6696
Epoch 50/50 | Ortalama Kayıp (Loss): 0.6664
Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.

--- TinyGemma Başlangıç Harfiyle Üretilen İsimler ---
Girdi: 'a' -> Üretilen İsim: a
Girdi: 'b' -> Üretilen İsim: bura
Girdi: 'e' -> Üretilen İsim: e
Girdi: 'm' -> Üretilen İsim: miray
Girdi: 's' -> Üretilen İsim: sed
Girdi: 'y' -> Üretilen İsim: yüs


### hiperparametre optimizasyonu yapalım

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Tüm isimleri aralarına başlangıç belirteci (<unk>) koyarak tokenize edelim
# Böylece model her ismin nerede başlayıp nerede bittiğini çok net öğrenecek.
all_token_ids = []
for word in words:
    # Her ismin başına özel bir başlangıç IDsi  ekliyoruz
    all_token_ids.append(vocab_dict["<unk>"])
    all_token_ids.extend(encode(word))

print(f"Yeni eğitilecek toplam token sayısı: {len(all_token_ids)}")

# 2. PyTorch Dataset Sınıfını Tanımlayalım
class NameDataset(Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        # block_size kadar geriye veri kalacak şekilde uzunluğu ayarla
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        # Girdi (X): block_size uzunluğunda token dizisi
        # Hedef (Y): Bir sonraki adıma kaydırılmış token dizisi
        x = torch.tensor(self.token_ids[idx : idx + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[idx + 1 : idx + self.block_size + 1], dtype=torch.long)
        return x, y

# Modelin bir seferde geriye dönük bakacağı pencere uzunluğu
block_size = 6
dataset = NameDataset(all_token_ids, block_size=block_size)

# DataLoader: Verileri batch'ler halinde modele besler
batch_size = 5
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Küçük bir kontrol yapalım
x_sample, y_sample = next(iter(dataloader))
print("Girdi Örneği (X) şekli:", x_sample.shape)  # [Batch_size, block_size]
print("Hedef Örneği (Y) şekli:", y_sample.shape)

Yeni eğitilecek toplam token sayısı: 1101
Girdi Örneği (X) şekli: torch.Size([5, 6])
Hedef Örneği (Y) şekli: torch.Size([5, 6])


In [15]:
import torch.nn as nn
import torch.optim as optim

# 1. Loss ve Optimizer tanımları
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

# 2. Eğitim Döngüsü
epochs = 100
model.train()

print("Eğitim Başlıyor...")
for epoch in range(epochs):
    epoch_loss = 0.0
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        # İleri geçiş
        outputs = model(x_batch)

        # EĞER çıktı bir tuple ise ilk elemanı al, değilse doğrudan kendisini al
        if isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs

        # Kaybı hesapla ve geriye yay
        loss = criterion(logits.view(-1, config.vocab_size), y_batch.view(-1))
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1:02d}/{epochs} | Ortalama Kayıp (Loss): {avg_loss:.4f}")

print("Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.\n")

# 3. İsim Üretme
model.eval()

def generate_perfect_start_name_gemma(start_char, max_len=10):
    # Modelin ismin başı olduğunu anlaması için girdinin başına <unk> (ID: 65 veya vocab_dict["<unk>"]) koyuyoruz
    input_ids = [vocab_dict["<unk>"]] + encode(start_char)

    # Ancak encode fonksiyonu sonuna </w> eklediği için onu temizleyelim ki kelime hemen bitmesin
    if input_ids[-1] == vocab_dict.get("</w>"):
        input_ids.pop()

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    generated_ids = list(input_ids)

    with torch.no_grad():
        for _ in range(max_len):
            outputs = model(input_tensor)

            if isinstance(outputs, tuple):
                logits = outputs[0][:, -1, :]
            else:
                logits = (outputs.logits if hasattr(outputs, 'logits') else outputs)[:, -1, :]

            probabilities = torch.softmax(logits / 0.8, dim=-1)
            next_token_id = torch.multinomial(probabilities, num_samples=1).item()

            # Eğer yeni bir isme geçmeye çalışıyorsa (<unk>) veya kelime bittiyse durdur
            if next_token_id == vocab_dict.get("<unk>") or next_token_id == vocab_dict.get("</w>"):
                break

            next_token_str = inverse_vocab.get(next_token_id, "")
            if "</w>" in next_token_str or "<unk>" in next_token_str:
                break

            generated_ids.append(next_token_id)

            input_tensor = torch.cat([input_tensor, torch.tensor([[next_token_id]]).to(device)], dim=1)
            if input_tensor.size(1) > block_size:
                input_tensor = input_tensor[:, -block_size:]

    # Başlangıç belirteçlerini temizle ve ekrana bas
    final_name = decode(generated_ids).replace("<unk>", "").strip()
    return final_name

# --- NİHAİ KUSURSUZ TEST ---
print("--- TinyGemma Başlangıç Harfiyle Üretilen İsimler ---")
for harf in ["a", "b", "e", "m", "s", "y"]:
    yeni_isim = generate_perfect_start_name_gemma(harf)
    print(f"Girdi: '{harf}' -> Üretilen İsim: {yeni_isim}")

Eğitim Başlıyor...
Epoch 01/100 | Ortalama Kayıp (Loss): 0.7767
Epoch 10/100 | Ortalama Kayıp (Loss): 0.7301
Epoch 20/100 | Ortalama Kayıp (Loss): 0.7092
Epoch 30/100 | Ortalama Kayıp (Loss): 0.6916
Epoch 40/100 | Ortalama Kayıp (Loss): 0.7025
Epoch 50/100 | Ortalama Kayıp (Loss): 0.6673
Epoch 60/100 | Ortalama Kayıp (Loss): 0.6621
Epoch 70/100 | Ortalama Kayıp (Loss): 0.6691
Epoch 80/100 | Ortalama Kayıp (Loss): 0.6511
Epoch 90/100 | Ortalama Kayıp (Loss): 0.6437
Epoch 100/100 | Ortalama Kayıp (Loss): 0.6686
Eğitim Tamamlandı! Model isimleri öğrenmeyi bitirdi.

--- TinyGemma Başlangıç Harfiyle Üretilen İsimler ---
Girdi: 'a' -> Üretilen İsim: a
Girdi: 'b' -> Üretilen İsim: belg
Girdi: 'e' -> Üretilen İsim: esarp
Girdi: 'm' -> Üretilen İsim: miraç
Girdi: 's' -> Üretilen İsim: sevil
Girdi: 'y' -> Üretilen İsim: yağmur
